# 01 — Feature Extraction and Creation

**Status:** `PENDING MANUAL EXECUTION`


## 1. Imports and Configuration


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
RANDOM_SEED = 42


## 2. Project Root and Path Configuration


In [ ]:
def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    raise RuntimeError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
INPUT_CSV = PROJECT_ROOT / "data" / "processed" / "charity-patient-realism-revised.csv"
OUT_DIR = PROJECT_ROOT / "data" / "interim" / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load Revised Experimental Dataset


In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows x {df.shape[1]} columns")


## 4. Inspect Available Columns


In [ ]:
REQUIRED_COLS = ["household_size", "monthly_family_income", "monthly_household_obligations_expenses", "total_amount", "eligibility"]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
print("Missing required columns:", missing)


## 5. Numeric Conversion Helper


In [ ]:
def to_num(col):
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")

hh = to_num("household_size").clip(lower=1)
inc = to_num("monthly_family_income")
exp = to_num("monthly_household_obligations_expenses")
cost = to_num("total_amount")


## 6. Feature Engineering Calculations


In [ ]:
df["eng_per_capita_income"] = inc / hh
df["eng_expense_to_income_ratio"] = exp / inc.replace(0, np.nan)
df["eng_treatment_to_income_ratio"] = cost / inc.replace(0, np.nan)


## 7. Feature Validation & Missingness Check


In [ ]:
eng_cols = ["eng_per_capita_income", "eng_expense_to_income_ratio", "eng_treatment_to_income_ratio"]
print(df[eng_cols].isna().sum())


## 8. Save Engineered Dataset


In [ ]:
out_path = OUT_DIR / "feature_engineered_dataset.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)
